# RD01-b — Residual regime discovery and online state confirmation

**Goal:** determine whether interpretable, duration-aware tree states explain next-5/15/30/60m *signed returns* beyond direct feature regression and explicit interactions. **Primary horizon: 30 minutes.**

Uses CS04-A causal data, RD01 chronology, and RD01's trailing variance proxy—not AM-T-HAR-Blend's forecast. Three-consecutive-bar state confirmation induces dwell mechanically and must **not** be treated as evidence of structural persistence. The repeatedly inspected historical periods are for discovery only.

Frozen protocol: `discussions/Regime_Discovery/001b_chatgpt_residual_regime_preregistration.md`.

In [ ]:
import os,sys,json,time,tempfile,subprocess,gc
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display
from dotenv import load_dotenv
ROOT=Path.cwd().resolve()
while ROOT!=ROOT.parent and not (ROOT/'src'/'data_engineering').is_dir():ROOT=ROOT.parent
assert (ROOT/'src'/'data_engineering').is_dir(), 'Open JupyterLab in your repository checkout'
sys.path.insert(0,str(ROOT/'src'))
load_dotenv(ROOT/'.env')
from data_engineering.modeling import observable_regime_discovery as rd
from data_engineering.modeling import observable_regime_discovery_b as rb
SYNTHETIC=os.getenv('RD01B_SYNTHETIC','0')=='1'
cfg=rb.Spec(minimum_bootstrap_days=5,bootstrap_repetitions=25,tree_min_leaf_absolute=25) if SYNTHETIC else rb.Spec()
if SYNTHETIC:
    temp=tempfile.TemporaryDirectory(prefix='rd01b-')
    OUT=Path(temp.name)/'01b'
else:
    rootval=os.getenv('DERIBIT_DATA_ROOT')
    if not rootval:raise RuntimeError('DERIBIT_DATA_ROOT not set')
    STORE=Path(rootval)/'research'/'cs04a_feature_store'
    OUT=ROOT/'notebooks'/'Regime_Discovery'/'01b'
OUT.mkdir(parents=True,exist_ok=True)
print('RD01-b',rb.VERSION,'synthetic',SYNTHETIC,'output',OUT)
env=os.environ.copy();env['PYTHONPATH']=str(ROOT/'src')+os.pathsep+env.get('PYTHONPATH','')
r=subprocess.run([sys.executable,'-m','pytest','-q',str(ROOT/'tests'/'test_observable_regime_discovery_b.py')],cwd=str(ROOT),env=env,text=True,capture_output=True)
print(r.stdout[-1300:]);print(r.stderr[-700:]);assert r.returncode==0, 'RD01-b tests failed; stop historical scan' 

In [ ]:
datasets={};excluded={}
if SYNTHETIC:
    sys.path.insert(0,str(ROOT/'tests'))
    from test_observable_regime_discovery import synthetic_panel
    datasets={'BTC':synthetic_panel(days=94)}
else:
    columns=list(dict.fromkeys(['state_time','return_5m','feature_ready','log_rv24h',*rd.BASE_COLS,*rd.TREE_COLS,'ret_5m','ret_15m','ret_30m']))
    for asset in cfg.assets:
        path=STORE/f'{asset}_features_and_forward_magnitude.parquet'
        if not path.is_file():excluded[asset]=f'Missing local feature store: {path}';continue
        try:
            datasets[asset]=pd.read_parquet(path,columns=columns)
            print(asset,'rows',len(datasets[asset]))
        except Exception as e:excluded[asset]=f'{type(e).__name__}: {e}'
assert datasets, f'No historical inputs: {excluded}'
if not SYNTHETIC and not {'BTC','ETH'}.issubset(datasets):print('WARNING: BTC/ETH primary coverage incomplete')

In [ ]:
results={};coverage=[]
for asset,frame in datasets.items():
    started=time.perf_counter()
    try:
        z=rb.scan_asset(frame,cfg,asset)
        results[asset]=z
        coverage.append({'asset':asset,'complete_days':z['n_days'],'score_rows':len(z['scores']),
                         'primary_tree_structures':len(z['definitions']),
                         'elapsed_seconds':round(time.perf_counter()-started,2)})
        print(asset,'complete',len(z['scores']),'scores',len(z['definitions']),'horizon tree structures')
    except Exception as e:
        excluded[asset]=f'{type(e).__name__}: {e}'
        if not SYNTHETIC:raise
    finally:
        del frame
        gc.collect()
assert results, 'No assets scanned'
display(pd.DataFrame(coverage))

In [ ]:
out=rb.consolidate(results,cfg)
out['manifest']['synthetic']=SYNTHETIC
out['manifest']['excluded']=excluded
out['manifest']['n_asset_folds']=len(out['scores'].groupby(['asset','fold']))
display(out['comparison_summary'].sort_values(['horizon_min','delta_nll_vs_direct'])[['method','horizon_min','n_asset_folds','delta_nll_vs_direct','delta_binary_vs_direct']])
print('30m per-asset-fold paired NLL against interaction Ridge:')
focus=out['bootstrap'].query("horizon_min==30 and metric=='nll' and reference=='interaction_ridge'")
display(focus[['asset','fold','method','delta','ci025','ci975','status']])
print('Persistence is mechanically influenced by the three-bar confirmation rule.')
display(out['persistence'][['asset','fold','method','observed_stay','within_day_iid_expected_stay']])
print('Descriptive decision only:',json.dumps(out['decision'],indent=2))

In [ ]:
outputs={'01b_scores.csv':'scores','01b_paired_bootstrap.csv':'bootstrap',
    '01b_comparison_summary.csv':'comparison_summary','01b_state_forecasts.csv':'characteristics',
    '01b_persistence.csv':'persistence','01b_episode_summary.csv':'episode_summary',
    '01b_transitions.csv':'transitions','01b_tree_definitions.csv':'definitions'}
for name,key in outputs.items():out[key].to_csv(OUT/name,index=False)
pd.DataFrame(coverage).to_csv(OUT/'01b_coverage.csv',index=False)
(OUT/'01b_manifest.json').write_text(json.dumps(out['manifest'],indent=2,default=str),encoding='utf-8')
(OUT/'01b_decision.json').write_text(json.dumps(out['decision'],indent=2,default=str),encoding='utf-8')
assert all((OUT/f).is_file() for f in list(outputs)+['01b_coverage.csv','01b_manifest.json','01b_decision.json'])
print('Saved',len(outputs)+3,'aggregate outputs to',OUT)
print('Synthetic outputs are NOT market evidence. Do not commit private feature store or per-origin labels.')

## Interpretation rules

- The main question is **incremental 30m signed-return density NLL versus interaction Ridge**, not whether sticky states persist (they are engineered to persist).
- Even a historical negative NLL delta is exploratory only. The legacy CS03b/VD18 heldout interval was previously examined.
- State labels are tree IDs, specific to **asset, fold and horizon**. No global regime names or duration claims are justified from these outputs.
- The common Student-t scale uses the RD01 trailing-RV proxy—not the AM-T-HAR-Blend conditional variance. Actual AM-T head-to-head belongs in RD03.
- The original RD01 code, previous research outputs, collectors and AM-T-HAR-Blend are unchanged.